In [8]:
import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from prettytable import PrettyTable
import sklearn
from sklearn.metrics import confusion_matrix

import time
from tensorflow import keras
import tensorflow as tf
from keras.models import Sequential
from keras.models import Sequential
from keras.layers import Dense, LSTM, Bidirectional, Flatten #Global max pullin avg max pulling
from keras.layers import  Masking
from keras.regularizers import l2
from tensorflow.keras import regularizers
from keras import callbacks
from keras.callbacks import EarlyStopping, ModelCheckpoint

import datetime

tf.config.list_physical_devices('GPU')

# fix random seed for reproducibility
seed = 42
tf.random.set_seed(seed)

In [9]:
def transformacao_estrutura(df, lista_variaveis):
    '''
    Extrai os atributos do dataframe pandas

    Params:
    -------
    df: dataframe de entrada

    Returns:
    par X, Y
    '''

    X = df[lista_variaveis]
    


    
    Y = df[['ocorrencia']]
    
    n_tempo, n_coord = conta_tempos_e_coordenadas(df)
    
    X_transformado = X.to_numpy().astype(np.float32).reshape(n_tempo , n_coord, -1)
    Y_transformado = Y.to_numpy().astype(np.float32).reshape(n_tempo , n_coord, -1)
    
    return X_transformado, Y_transformado

    
def conta_tempos_e_coordenadas(df):
    '''
    Conta o número de tempos e coordenadas de um dataframe

    Params:
    -------
    df: dataframe com os atributos 'tempo', e 'n_coord'

    Returns:
    --------
    tupla número de tempos, número de coordenadas
    '''
    tempo = len(df['tempo'].unique())
    n_coord = len(df['n_coord'].unique())

    return tempo, n_coord


def calcula_sample_weight(df,peso):
    '''
    Calcula os pesos para as amostras de treinamento
    '''
    
    Y = df[['ocorrencia']]
    sample_weight_ = list(Y['ocorrencia'])
    
    series = pd.Series(sample_weight_)
    series = series*peso #3191848/613
    series = series+1
    series = series/(peso+1)
    
    sample_weight_ = np.array(list(series)).astype(np.float32)
    
    return sample_weight_
    
    
def flatten(l):
    return [item for sublist in l for item in sublist]    



In [10]:
# # Importacao dos dados

# In[ ]:


df = pd.read_csv('df_comp2.csv')


# In[ ]:


df.replace(np.nan,-1, inplace=True)


# In[ ]:


df['tempo'] =  pd.to_datetime(df['tempo'])


# In[ ]:


df = df.sort_values(['tempo','n_coord'])


# In[ ]:


lista_variaveis = [
     'vintequatrohoras'                                                              
    ,'Declividade_numerica'                                                                 
    ,'Orientacao_numerica'                             
    ,'Curv_Vertical_numerica'                          
    ,'Curv_Horizontal_numerica'                        
    ,'Relevo_sombreado_numerico'                       
    ,'Vegetacao_Natural_Dominante'                     
    ,'Area_Antropica_Dominante'                        
    ,'legenda_2_Pecuária (pastagens)'                                                                            
    ,'Influencia_urbana'                               
    ,'Vegetacao_Secundaria'                            
    ,'Argilossolo'                                     
    ,'Gleissolo'                                       
    ,'Argilossolo_Vermelho_Amarelo'                    
    ,'Gleissolo_Melanico'                              
    ,'Area_Urbana'                                     
    ,'Unidades_de_Conservacao_Protecao_Integral'                                      
    ,'flg_comunidades'                                 
    ,'flg_agricola'                                    
    ,'flg_exploracao_mineral'                          
    ,'flg_rocha'                                       
    ,'flg_cobertura_vegetal'                           
    ,'flg_afloramento_rochoso'                                                              
    ,'flg_ocupacao_desordenada'                         
           ]


# Manter fixo:
# 
# * Teste  - 6 meses
# 
# Deslizar:
# 
# * Treinamento - 2 anos
# 
# * Validação - 6 meses

# In[ ]:


janela_treino = ['2019-02-01 09:00:00'
                ,'2019-03-01 09:00:00'
                ,'2019-04-01 09:00:00'
                ,'2019-05-01 09:00:00'
                ,'2019-06-01 09:00:00'
                ,'2019-07-01 09:00:00'
                ,'2019-08-01 09:00:00'
                ,'2019-09-01 09:00:00'
                ,'2019-10-01 09:00:00'
                ,'2019-11-01 09:00:00'
                ,'2019-12-01 09:00:00'
                ,'2020-01-01 09:00:00'
                ,'2020-02-01 09:00:00'
                ,'2020-03-01 09:00:00'
                ]

janela_validacao_i = ['2021-02-01 09:00:00'
                ,'2021-03-01 09:00:00'
                ,'2021-04-01 09:00:00'
                ,'2021-05-01 09:00:00'
                ,'2021-06-01 09:00:00'
                ,'2021-07-01 09:00:00'
                ,'2021-08-01 09:00:00'
                ,'2021-09-01 09:00:00'
                ,'2021-10-01 09:00:00'
                ,'2021-11-01 09:00:00'
                ,'2021-12-01 09:00:00'
                ,'2022-01-01 09:00:00'
                ,'2022-02-01 09:00:00'
                ,'2022-03-01 09:00:00'
                ]

janela_validacao_s = ['2021-08-01 09:00:00'
                ,'2021-09-01 09:00:00'
                ,'2021-10-01 09:00:00'
                ,'2021-11-01 09:00:00'
                ,'2021-12-01 09:00:00'
                ,'2022-01-01 09:00:00'
                ,'2022-02-01 09:00:00'
                ,'2022-03-01 09:00:00'
                ,'2022-04-01 09:00:00'
                ,'2022-05-01 09:00:00'
                ,'2022-06-01 09:00:00'
                ,'2022-07-01 09:00:00'
                ,'2022-08-01 09:00:00'
                ,'2022-09-01 09:00:00'
                ]

In [11]:
janela_i = 1
i = janela_i-1

filtro_inferior = pd.to_datetime(janela_treino[i]) <= df.loc[:, 'tempo']
filtro_superior = df.loc[:,'tempo'] < pd.to_datetime(janela_validacao_i[i])
treino = df[filtro_inferior & filtro_superior]

filtro_inferior = pd.to_datetime(janela_validacao_i[i]) <= df.loc[:, 'tempo']
filtro_superior = df.loc[:, 'tempo'] < pd.to_datetime(janela_validacao_s[i])
validacao = df[filtro_inferior & filtro_superior]

teste = df[df.loc[:,'tempo'] >= pd.to_datetime('2022-09-01 09:00:00')]

### Treino

X_train, Y_train = transformacao_estrutura(treino, lista_variaveis)

### Validação

X_val, Y_val = transformacao_estrutura(validacao, lista_variaveis)

### Teste

X_teste, Y_teste = transformacao_estrutura(teste, lista_variaveis)


print('Treino Ocorrências, #Treino, Validação Ocorrências, #Validação, Teste Ocorrências, #Teste')
T_qtd_o = sum(treino['ocorrencia'])
T_tam_o = len(treino['ocorrencia'])
peso = T_tam_o / T_qtd_o


V_qtd_o = sum(validacao['ocorrencia'])
V_tam_o = len(validacao['ocorrencia'])

Tes_qtd_o = sum(teste['ocorrencia'])
Tes_tam_o = len(teste['ocorrencia'])

print(f'{T_qtd_o},{T_tam_o},{V_qtd_o},{V_tam_o},{Tes_qtd_o},{Tes_tam_o}')


sample_weight_ = np.asarray(calcula_sample_weight(treino,round(peso))).astype(np.float32)

sample_weight_transformado = sample_weight_.reshape(len(treino['tempo'].unique()), len(treino['n_coord'].unique()), -1)


##################### Modelo ##################

time_step = X_train.shape[1] # Quantidade de coordenada para equivaler a 1 espaço de tempo
input_dim = X_train.shape[2] #qtd colunas (features)
out = Y_train.shape[2]

# LSTM
start = time.time()
model = Sequential()
model.add(Masking(mask_value=-1.,input_shape=(time_step, input_dim,))) #camada de entrada
model.add(LSTM(64,activation='sigmoid'
                     , input_shape=(time_step, input_dim)
                     ,return_sequences=True
                     , go_backwards=True
                     #, kernel_regularizer=regularizers.L1L2(l1=1e-5, l2=1e-4)
              )) # camada escondida
model.add(Dense(out, activation='sigmoid')) #camada saida

opt = tf.keras.optimizers.Adam(learning_rate=0.01
                               #, clipnorm=1
                               #,clipvalue=0.5
                              )

model.compile(loss = tf.keras.losses.BinaryCrossentropy(from_logits=False) #https://keras.io/api/losses/probabilistic_losses/ 
              , optimizer= opt #'adam'
              , weighted_metrics=[tf.keras.losses.BinaryCrossentropy(from_logits=False),'accuracy'
              , tf.keras.metrics.Precision()
              , tf.keras.metrics.Recall()]
             , sample_weight_mode="temporal"   #Weights
             )   
model.summary()
hist = model.fit(X_train
                 , Y_train
                 , epochs=100
                 , validation_data=(X_val, Y_val)
                 , verbose=1
                 , batch_size=64
                 , sample_weight=sample_weight_transformado  #(samples, sequence_length)
        )
end = time.time()
print("Total compile time: --------", end - start, 's')

predictions = model.predict(X_teste, verbose=1).reshape((1,-1))

pred_y = pd.DataFrame(flatten(predictions), columns=['Prediction']) 




Treino Ocorrências, #Treino, Validação Ocorrências, #Validação, Teste Ocorrências, #Teste
545.0,2247795,9.0,567435,217.0,564300
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 masking_1 (Masking)         (None, 3135, 24)          0         
                                                                 
 lstm_1 (LSTM)               (None, 3135, 64)          22784     
                                                                 
 dense_1 (Dense)             (None, 3135, 1)           65        
                                                                 
Total params: 22,849
Trainable params: 22,849
Non-trainable params: 0
_________________________________________________________________
Epoch 1/100
12/12 [==============================] - 17s 1s/step - loss: 3.2166e-04 - binary_crossentropy: 0.6257 - accuracy: 0.6055 - precision_1: 0.6381 - recall_1: 0.4881 - val_loss: 

12/12 [==============================] - 16s 1s/step - loss: 1.9057e-04 - binary_crossentropy: 0.3878 - accuracy: 0.8260 - precision_1: 0.8375 - recall_1: 0.8092 - val_loss: 0.4223 - val_binary_crossentropy: 0.4223 - val_accuracy: 0.8124 - val_precision_1: 8.4542e-05 - val_recall_1: 1.0000
Epoch 25/100
12/12 [==============================] - 16s 1s/step - loss: 1.9226e-04 - binary_crossentropy: 0.4106 - accuracy: 0.8236 - precision_1: 0.8282 - recall_1: 0.8165 - val_loss: 0.4199 - val_binary_crossentropy: 0.4199 - val_accuracy: 0.8397 - val_precision_1: 9.8927e-05 - val_recall_1: 1.0000
Epoch 26/100
12/12 [==============================] - 16s 1s/step - loss: 1.9564e-04 - binary_crossentropy: 0.3832 - accuracy: 0.8286 - precision_1: 0.8447 - recall_1: 0.8055 - val_loss: 0.3727 - val_binary_crossentropy: 0.3727 - val_accuracy: 0.8506 - val_precision_1: 1.0612e-04 - val_recall_1: 1.0000
Epoch 27/100
12/12 [==============================] - 16s 1s/step - loss: 1.9409e-04 - binary_crossen

Epoch 51/100
12/12 [==============================] - 16s 1s/step - loss: 1.6439e-04 - binary_crossentropy: 0.3244 - accuracy: 0.8507 - precision_1: 0.8649 - recall_1: 0.8312 - val_loss: 0.2738 - val_binary_crossentropy: 0.2738 - val_accuracy: 0.8923 - val_precision_1: 1.1454e-04 - val_recall_1: 0.7778
Epoch 52/100
12/12 [==============================] - 16s 1s/step - loss: 1.6073e-04 - binary_crossentropy: 0.3231 - accuracy: 0.8554 - precision_1: 0.8651 - recall_1: 0.8422 - val_loss: 0.2999 - val_binary_crossentropy: 0.2999 - val_accuracy: 0.8650 - val_precision_1: 1.0446e-04 - val_recall_1: 0.8889
Epoch 53/100
12/12 [==============================] - 16s 1s/step - loss: 1.6545e-04 - binary_crossentropy: 0.3379 - accuracy: 0.8525 - precision_1: 0.8547 - recall_1: 0.8495 - val_loss: 0.3059 - val_binary_crossentropy: 0.3059 - val_accuracy: 0.8809 - val_precision_1: 1.1838e-04 - val_recall_1: 0.8889
Epoch 54/100
12/12 [==============================] - 16s 1s/step - loss: 1.7376e-04 - b

Epoch 78/100
12/12 [==============================] - 16s 1s/step - loss: 1.6204e-04 - binary_crossentropy: 0.3021 - accuracy: 0.8525 - precision_1: 0.8712 - recall_1: 0.8275 - val_loss: 0.2611 - val_binary_crossentropy: 0.2611 - val_accuracy: 0.8981 - val_precision_1: 1.2109e-04 - val_recall_1: 0.7778
Epoch 79/100
12/12 [==============================] - 16s 1s/step - loss: 1.5951e-04 - binary_crossentropy: 0.3022 - accuracy: 0.8534 - precision_1: 0.8727 - recall_1: 0.8275 - val_loss: 0.2545 - val_binary_crossentropy: 0.2545 - val_accuracy: 0.8947 - val_precision_1: 1.3391e-04 - val_recall_1: 0.8889
Epoch 80/100
12/12 [==============================] - 16s 1s/step - loss: 1.5511e-04 - binary_crossentropy: 0.3222 - accuracy: 0.8598 - precision_1: 0.8633 - recall_1: 0.8550 - val_loss: 0.1978 - val_binary_crossentropy: 0.1978 - val_accuracy: 0.9273 - val_precision_1: 1.4550e-04 - val_recall_1: 0.6667
Epoch 81/100
12/12 [==============================] - 16s 1s/step - loss: 1.5319e-04 - b

In [12]:
#analise = teste[['tempo', 'n_coord',  'vintequatrohoras', 'ocorrencia']]

In [13]:
analise.insert(len(analise.columns), 'Prediction', pred_y['Prediction'].values)

In [14]:
analise = analise.rename(columns={"Prediction": "Prediction_M3J1E2"})

In [15]:
analise.to_csv('df_analise_prob.csv')

In [7]:
#analise  = pd.read_csv('df_analise_prob.csv')

In [16]:
analise

,Unnamed: 0,tempo,n_coord,vintequatrohoras,ocorrencia,Prediction_M3J1E1,Prediction_M3J1E2,Prediction_M6J13E1,Prediction_M6J13E2,Prediction_M3J1E2
0,3679321,2022-09-01 09:00:00,0,-11.512925,0.0,0.241371,0.221999,0.337923,0.456830,0.600053
1,3680394,2022-09-01 09:00:00,1,-11.512925,0.0,0.267321,0.287599,0.388327,0.384408,0.598111
2,3680393,2022-09-01 09:00:00,2,-11.512925,0.0,0.151297,0.189529,0.329339,0.370375,0.486485
3,3680392,2022-09-01 09:00:00,3,-11.512925,0.0,0.105065,0.154497,0.329981,0.322659,0.414955
4,4445096,2022-09-01 09:00:00,4,3.508556,0.0,0.115740,0.186915,0.301187,0.351039,0.507715
...,...,...,...,...,...,...,...,...,...,...
564295,4184452,2023-02-27 09:00:00,3130,-11.512925,0.0,0.992259,0.975510,0.733034,0.734852,0.979384
564296,4501858,2023-02-27 09:00:00,3131,-11.512925,0.0,0.979717,0.967006,0.750197,0.686308,0.959488
564297,4501859,2023-02-27 09:00:00,3132,-11.512925,0.0,0.977374,0.961052,0.732363,0.663485,0.952956
564298,4184804,2023-02-27 09:00:00,3133,-11.512925,0.0,0.977509,0.963485,0.743006,0.673677,0.951979
